# ChaosOps AI — Colab training notebook

Minimum-viable GRPO training run for the **ChaosOps AI** multi-agent incident-response environment with rogue-agent detection.

**What this notebook does (top-to-bottom, <30 min on a free T4):**
1. Installs deps (Unsloth 4-bit, TRL GRPO, transformers, peft)
2. Clones the project from GitHub
3. Runs **scripted baselines** (random / heuristic / oracle) → produces `baseline_curve.png`
4. Runs a **small GRPO training pass** on Qwen 2.5 0.5B to show the reward curve moves in the right direction
5. Plots the training metrics

> T4-friendly settings: Qwen 2.5 **0.5B** Instruct, 4-bit quant, LoRA r=16, 30 episodes, group size 2, max_seq_length=1024. The real hackathon training run uses **Qwen 2.5 7B** + LoRA r=32 on onsite HF-credit GPUs.

**Pitch materials produced:**
- `artifacts/baseline/baseline_curve.png` — Random vs. Heuristic vs. Oracle reward by tier
- `artifacts/chaosops-grpo/training_metrics.json` + rendered learning curve
- LoRA checkpoint in `artifacts/chaosops-grpo/`

In [ ]:
# 1. Verify GPU
!nvidia-smi | head -n 20

## 1 · Install dependencies

Unsloth handles the 4-bit loader + LoRA attach; TRL provides GRPOTrainer.

In [ ]:
%%capture
!pip install --upgrade --quiet \
    "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" \
    "trl>=0.11.0" \
    "transformers>=4.44" \
    "peft>=0.12" \
    "accelerate>=0.33" \
    "bitsandbytes>=0.43" \
    "datasets" \
    "rich" \
    "pydantic>=2" \
    "matplotlib"

## 2 · Clone ChaosOps AI

In [ ]:
import os
if not os.path.exists('chaos_ops'):
    !git clone https://github.com/vatsalllll/chaos_ops.git
%cd chaos_ops
!ls

In [ ]:
# Put project on PYTHONPATH
import sys, os
sys.path.insert(0, os.getcwd())
import chaosops
print("chaosops package loaded from:", chaosops.__file__)

## 3 · Scripted baselines (the "before training" numbers)

This is the numeric evidence for the **Reward Improvement** rubric criterion. Runs in <2 min on CPU — no GPU needed.

In [ ]:
!python -m chaosops.train.baseline --episodes-per-type 5

In [ ]:
from IPython.display import Image
Image("artifacts/baseline/baseline_curve.png")

## 4 · GRPO training pass (Qwen 2.5 0.5B, T4-sized)

Small model / few episodes so the notebook completes inside the free-tier runtime budget. The pipeline is identical to the full onsite run — just smaller knobs.

In [ ]:
!python -m chaosops.train.grpo_train \
    --model-name Qwen/Qwen2.5-0.5B-Instruct \
    --total-episodes 30 \
    --group-size 2 \
    --team-weight 0.6 \
    --log-every 2 \
    --max-seq-length 1024 \
    --lora-rank 16 \
    --output-dir artifacts/chaosops-grpo \
    --start-tier easy

## 5 · Plot the learning curve

In [ ]:
import json, matplotlib.pyplot as plt
from pathlib import Path

metrics_path = Path('artifacts/chaosops-grpo/training_metrics.json')
metrics = json.loads(metrics_path.read_text())

episodes = [m['episode'] for m in metrics]
team = [m.get('mean_team_reward', float('nan')) for m in metrics]
oversight = [m.get('mean_oversight_reward', float('nan')) for m in metrics]
combined = [m.get('mean_combined_reward', float('nan')) for m in metrics]

fig, ax = plt.subplots(figsize=(9, 4.5), dpi=150)
ax.plot(episodes, team, label='team reward', color='#2980b9', linewidth=2)
ax.plot(episodes, oversight, label='oversight reward', color='#c0392b', linewidth=2)
ax.plot(episodes, combined, label='combined (0.6·team + 0.4·oversight)', color='#27ae60', linewidth=2, linestyle='--')
ax.axhline(0, color='#888', linewidth=0.6)
ax.set_xlabel('episode')
ax.set_ylabel('mean reward over group')
ax.set_title('ChaosOps AI — GRPO learning curve (Qwen 2.5 0.5B, Colab T4)')
ax.legend(loc='lower right')
ax.grid(True, linestyle=':', alpha=0.4)
plt.tight_layout()
plt.savefig('artifacts/chaosops-grpo/learning_curve.png')
plt.show()

## 6 · Next steps (onsite, HF-credit GPU)

Once you get bigger GPUs, bump the knobs:

```bash
python -m chaosops.train.grpo_train \
    --model-name Qwen/Qwen2.5-7B-Instruct \
    --total-episodes 300 \
    --group-size 4 \
    --team-weight 0.6 \
    --log-every 5 \
    --max-seq-length 1536 \
    --lora-rank 32 \
    --output-dir artifacts/chaosops-grpo-7b \
    --start-tier easy
```

Expected outcome for the pitch slide:
- MTTR on HARD: **~10 steps → ~4 steps**
- Rogue-catch rate: **~20% → >70%**
- Mean HARD reward: **−237 → >+50**